In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import pandas as pd
import time
import os

# Load TenderNo list from Excel
df = pd.read_excel("All_data_live_tender.xlsx")
tender_nos = df['TenderNo'].dropna().unique()

# Setup Chrome
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

# Folder to save links
os.makedirs("boq_links", exist_ok=True)
log_file = open("boq_links/found_links.txt", "w", encoding="utf-8")

for tender_no in tender_nos[:100]:  # limit to 100 initially for testing
    search_query = f'site:sites.google.com OR site:drive.google.com "{tender_no}" BOQ'
    driver.get("https://www.google.com/")
    time.sleep(2)

    # Accept cookies (only if needed)
    try:
        driver.find_element(By.XPATH, '//button[text()="Accept all"]').click()
        time.sleep(1)
    except:
        pass

    # Perform search
    search_box = driver.find_element(By.NAME, "q")
    search_box.clear()
    search_box.send_keys(search_query)
    search_box.send_keys(Keys.RETURN)
    time.sleep(3)

    try:
        # Capture first few result links
        results = driver.find_elements(By.XPATH, '//div[@class="g"]//a')
        for res in results[:3]:  # first 3 links
            link = res.get_attribute('href')
            if any(ext in link for ext in ['.xls', '.xlsx', '.pdf', '.docx']):
                log_file.write(f"{tender_no} : {link}\n")
                print(f"Found: {tender_no} → {link}")
                break
    except Exception as e:
        print(f"Error for {tender_no}: {e}")

log_file.close()
driver.quit()
